# Étape 5 : table des anomalies

Ce notebook réunit dans une seule table les écarts des trois contrôles : format et complétude, doublons (notebook 04), règles de gestion (notebook 05). Chaque ligne garde la maille de sa source (cellule, ligne, ou période pour un total), son lignage, la clé de sa ligne, la règle violée, le constat et l'issue (anomalie, à arbitrer ou information), avec un identifiant stable d'une exécution à l'autre. Un bilan donne, par contrôle, les lignes contrôlées, les anomalies, le taux d'anomalies et le nombre de lignes sources en anomalie. Rien n'est corrigé. La dernière cellule écrit la table et le bilan en CSV dans `data/anomalies/` (`anomalies_AAAA-MM.csv` et `bilan_AAAA-MM.csv`, réécrits à chaque exécution); pour les ouvrir dans Excel sans altérer les valeurs, passer par Données, À partir d'un fichier texte/CSV, et régler la détection du type de données sur « Ne pas détecter les types de données ». Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `03_dictionnaire`, puis celui-ci en entier. Il relance lui-même les contrôles des notebooks 04 et 05, qui restent utiles pour le détail de chaque contrôle.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 03_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
%%time
# CONTROLES

from frigeo.controles import charger_regles_complementaires, controler
from frigeo.doublons import charger_declarations_doublons, detecter_doublons
from frigeo.gestion import charger_regles_gestion, controler_gestion

# Les contrôles des notebooks 04 et 05, relancés sur les mêmes données.
declarations = charger_declarations_doublons()
ecarts, bilan = controler(tables, dictionnaire, complementaires=charger_regles_complementaires())
doublons, bilan_doublons = detecter_doublons(tables, declarations)
ecarts_gestion, bilan_gestion = controler_gestion(
    tables, dictionnaire, charger_regles_gestion(), declarations, doublons,
    date_coupure=perimetre.date_coupure,
)
print(len(ecarts), "écarts de format,", len(doublons), "lignes en doublon,",
      len(ecarts_gestion), "écarts de gestion")
print(len(bilan), "contrôles de format,", len(bilan_doublons), "contrôles de doublons,",
      len(bilan_gestion), "règles de gestion")

In [ ]:
# ANOMALIES

from frigeo.anomalies import anomalies_non_masquees, reunir_anomalies

anomalies = reunir_anomalies(
    tables, ecarts, doublons, ecarts_gestion, bilan, bilan_doublons, bilan_gestion, declarations
)
print(len(anomalies), "lignes dans la table des anomalies,",
      anomalies["id_anomalie"].nunique(), "identifiants distincts")
display(anomalies.groupby(["etape", "issue"], sort=False).size().rename("lignes").to_frame())
print(int(anomalies["num_ligne_source"].isna().sum()), "écarts sans lignage (totaux par période)")
print(int(anomalies["cle"].isna().sum()), "lignes sans clé (table sans clé déclarée)")
# Aucune valeur ni clé visible sur une colonne sensible (attendu 0).
print(len(anomalies_non_masquees(anomalies, declarations=declarations)),
      "lignes dont la valeur ou la clé sensible n'est pas masquée")

In [ ]:
# BILAN PAR CONTROLE

from frigeo.anomalies import bilan_controles

bilan_anomalies = bilan_controles(bilan, bilan_doublons, bilan_gestion, anomalies, declarations)
print(len(bilan_anomalies), "contrôles, dont",
      int((bilan_anomalies["anomalies"] > 0).sum()), "avec au moins une anomalie")
display(
    bilan_anomalies.groupby("etape", sort=False)[
        ["controlees", "anomalies", "a_arbitrer", "informations"]
    ].sum()
)
# Contrôles qui ont des écarts, du plus fort taux d'anomalies au plus faible
# (taux vide pour un total par période).
avec_ecarts = bilan_anomalies[
    bilan_anomalies[["anomalies", "a_arbitrer", "informations"]].sum(axis=1) > 0
]
display(
    avec_ecarts.drop(columns=["gravite", "indicateurs"])
    .assign(taux=lambda d: pd.to_numeric(d["taux"]).round(4))
    .sort_values("taux", ascending=False, na_position="last")
)

In [ ]:
# GRAVITE ET INDICATEURS

from frigeo.anomalies import (
    avertissements_qualification,
    charger_qualification,
    chemin_qualification,
    colonnes_non_declarees,
    qualifier,
)
from frigeo.controles import ISSUE_ANOMALIE

# Sans fichier, rien n'est déclaré et tous les contrôles sortent non qualifiés.
qualification = charger_qualification()
print(len(qualification.get("indicateurs") or {}), "indicateurs,",
      sum(len(c or {}) for c in (qualification.get("colonnes") or {}).values()),
      "colonnes déclarées,", len(qualification.get("exceptions") or []),
      "exceptions, lus dans", chemin_qualification())
anomalies, bilan_anomalies, non_qualifies = qualifier(
    anomalies, bilan_anomalies, qualification, tables
)
avertissements = avertissements_qualification(bilan_anomalies, qualification)
print(len(avertissements), "avertissements")
for message in avertissements:
    print("  ", message)
# Anomalies seules (ni écarts à arbitrer ni informations), par étape et par gravité.
en_anomalie = anomalies[anomalies["issue"] == ISSUE_ANOMALIE]
display(
    pd.crosstab(en_anomalie["etape"], en_anomalie["gravite"].fillna("non qualifié"))
    .reindex([e for e in ("format", "doublons", "gestion") if e in set(en_anomalie["etape"])])
    .rename_axis(index="etape", columns="gravite")
)
print(len(non_qualifies), "contrôles non qualifiés (gravité et indicateurs vides)")
if len(non_qualifies):
    display(non_qualifies)
non_declarees = colonnes_non_declarees(tables, qualification)
print(len(non_declarees), "colonnes non déclarées dans config/anomalies.yaml")
if len(non_declarees):
    display(non_declarees.groupby("table", sort=False)["colonne"].agg(", ".join).to_frame())

In [ ]:
# ECRITURE

from frigeo.anomalies import dossier_anomalies, ecrire_anomalies

# Réécrit les deux fichiers de la période dans data/anomalies, sans copie de sauvegarde.
# En cas de refus (valeur sensible visible, table non qualifiée), rien n'est écrit.
# Si un des fichiers est ouvert dans Excel, le fermer puis relancer cette cellule.
chemin_anomalies, chemin_bilan = ecrire_anomalies(
    anomalies, bilan_anomalies, PERIODE_FIN, declarations=declarations
)
print("fichiers écrits dans", dossier_anomalies())
print("  ", f"{chemin_anomalies.name},", len(anomalies), "lignes")
print("  ", f"{chemin_bilan.name},", len(bilan_anomalies), "contrôles")